In [ ]:
from pyattck import Attck

# 初始化ATT&CK数据集
attack = Attck()
techniques = attack.enterprise.techniques
tactics = attack.enterprise.tactics

# 打印技术示例
print(techniques[0].name)  # 输出：'Access Token Manipulation'
print(techniques[0].id)    # 输出：'T1134'

In [5]:
from rapidfuzz import process, fuzz

def map_text_to_technique(text, techniques):
    """
    将文本关键词映射到ATT&CK技术（支持别名模糊匹配）
    """
    # 提取所有技术名称和别名
    tech_names = []
    for tech in techniques:
        tech_names.append(tech.name.lower())
        if hasattr(tech, 'aliases'):
            tech_names.extend([alias.lower() for alias in tech.aliases])
    
    # 模糊匹配（取相似度最高）
    matched, score, _ = process.extractOne(text.lower(), tech_names, scorer=fuzz.token_set_ratio)
    
    if score > 80:  # 相似度阈值设为80%
        for tech in techniques:
            if matched == tech.name.lower() or (hasattr(tech, 'aliases') and matched in [a.lower() for a in tech.aliases]):
                return tech
    return None

In [ ]:
def build_attack_matrix_from_text(text,keywords):
      
    # 提取技术实体（此处简化，实际需要ATTCK框架支持模型）
    techniques = ["powershell", "lsass", "凭据窃取"]
    
    # 映射到ATT&CK技术
    results = []
    for keyword in keywords:
        tech = map_text_to_technique(keyword, techniques)
        if tech:
            results.append({
                "keyword": keyword,
                "technique_id": tech.id,
                "technique_name": tech.name,
                "tactics": [tactic.name for tactic in tech.tactics]
            })
    
    # 生成矩阵视图（按战术分类）
    matrix = {}
    for item in results:
        for tactic in item['tactics']:
            if tactic not in matrix:
                matrix[tactic] = []
            matrix[tactic].append({
                "technique_id": item['technique_id'],
                "keyword": item['keyword']
            })
    return matrix


In [9]:
#实体提取模型
# Load model directly
from transformers import AutoTokenizer, AutoModelForTokenClassification
model_dir = '../../models/ner_model/'
tokenizer = AutoTokenizer.from_pretrained(model_dir+'bert-base-NER')
model = AutoModelForTokenClassification.from_pretrained(model_dir+'bert-base-NER')

Some weights of the model checkpoint at ../../models/ner_model/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [13]:
from transformers import pipeline
# 加载预训练模型
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)
text = "APT29 uses T1059 to execute malicious scripts."
entities = ner_pipeline(text)  # 提取实体

print(entities)

Device set to use cpu


[{'entity': 'B-ORG', 'score': np.float32(0.8447797), 'index': 1, 'word': 'AP', 'start': 0, 'end': 2}, {'entity': 'I-ORG', 'score': np.float32(0.41471845), 'index': 2, 'word': '##T', 'start': 2, 'end': 3}, {'entity': 'I-ORG', 'score': np.float32(0.5367049), 'index': 3, 'word': '##29', 'start': 3, 'end': 5}]


In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# 指定本地模型路径
tokenizer = AutoTokenizer.from_pretrained(model_dir+'rebel-large')
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir+'rebel-large')

In [20]:
from transformers import pipeline


rel_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=256
)

text = "Attackers heavily focused on acquiring military and security intelligence in order to support invading forces. The Shuckworm espionage group is continuing to mount multiple cyber attacks against Ukraine, with recent targets including security services, military, and government organizations. In some cases, Shuckworm has succeeded in staging long-running intrusions, lasting for as long as three months. The attackers repeatedly attempted to access and steal sensitive information such as reports about the deaths of Ukrainian military service members, enemy engagements and air strikes, arsenal inventories, military training, and more. In a bid to stay ahead of detection, Shuckworm has repeatedly refreshed its toolset, rolling out new versions of known tools and short-lived infrastructure, along with new additions, such as USB propagation malware."

relations = rel_pipeline(text)  # 生成三元组（主体，关系，客体）

print(rel_pipeline(text))

Device set to use cpu


[{'generated_text': ' Shuckworm  malware  instance of  Shuckworm  malware  instance of'}]
